# SimpleText Task 1.1 — Semantic Selector V4 (RAG + MedSimplify)




In [1]:
import os
import re
import ast
import sys
import json
import time
import random
import subprocess
from pathlib import Path

import numpy as np
import pandas as pd
import requests
from tqdm.auto import tqdm

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
pd.set_option("display.max_colwidth", 300)

print("✓ Imports ready")

✓ Imports ready


## Paths + Config

In [2]:
TRAIN_PATH         = Path("/kaggle/input/datasets/raneemrabi3/cochraneauto-sents/cochraneauto_sents_train.csv")
VALID_PATH         = Path("/kaggle/input/datasets/raneemrabi3/cochraneauto-sents/cochraneauto_sents_val.csv")
TEST_INTERNAL_PATH = Path("/kaggle/input/datasets/raneemrabi3/cochraneauto-sents/cochraneauto_sents_test.csv")
OFFICIAL_TEST_JSON = Path("/kaggle/input/datasets/raneemrabi3/testjson/simpletext25_task11_test (1).json")
MEDSIMPLIFY_PATH   = Path("/kaggle/input/datasets/raneemrabi3/medsimplify/MedSimplify.csv")

PROJECT_DIR = Path("/kaggle/working/simpletext_v4_rag")
OUTPUT_DIR  = PROJECT_DIR / "outputs"
PROJECT_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

RUN_ID      = "semantic_selector_v4_rag"
SPLIT_LIMIT = 200
SAVE_EVERY  = 25

# API
API_TIMEOUT          = 40
API_MAX_RETRIES      = 5
API_BACKOFF_BASE     = 4
API_RATE_LIMIT_SLEEP = 1.0
REQUEST_SLEEP_SEC    = 0.2
MAX_NEW_TOKENS       = 120
API_TEMPERATURE      = 0.0


FKGL_SIMPLE_THRESHOLD = 10.0


RAG_MAX_TERMS = 

print("PROJECT_DIR:", PROJECT_DIR)
print("OUTPUT_DIR :", OUTPUT_DIR)

PROJECT_DIR: /kaggle/working/simpletext_v4_rag
OUTPUT_DIR : /kaggle/working/simpletext_v4_rag/outputs


## API Setup

In [3]:
try:
    from kaggle_secrets import UserSecretsClient
    user_secrets = UserSecretsClient()
    GROQ_API_KEY = user_secrets.get_secret("API_INTERFACE")
except Exception:
    GROQ_API_KEY = os.getenv("API_INTERFACE", "")

GROQ_API_URL = "https://api.groq.com/openai/v1/chat/completions"

GROQ_MODEL_PLANNER  = "meta-llama/llama-4-scout-17b-16e-instruct"
GROQ_MODEL_EXECUTOR = "meta-llama/llama-4-scout-17b-16e-instruct"

print("GROQ loaded:", bool(GROQ_API_KEY))

def call_llama_api(prompt: str,
                   max_tokens: int = MAX_NEW_TOKENS,
                   use_large: bool = False) -> str:
    if not GROQ_API_KEY:
        raise ValueError("GROQ_API_KEY not found")

    model = GROQ_MODEL_PLANNER if use_large else GROQ_MODEL_EXECUTOR

    headers = {
        "Authorization": f"Bearer {GROQ_API_KEY}",
        "Content-Type": "application/json",
    }
    payload = {
        "model": model,
        "messages": [{"role": "user", "content": prompt}],
        "max_tokens": max_tokens,
        "temperature": API_TEMPERATURE,
        "stream": False,
    }

    for attempt in range(1, API_MAX_RETRIES + 1):
        try:
            resp = requests.post(
                GROQ_API_URL, headers=headers,
                json=payload, timeout=API_TIMEOUT
            )
            if resp.status_code == 429:
                time.sleep(API_RATE_LIMIT_SLEEP)
                continue
            if resp.status_code >= 500:
                time.sleep(API_BACKOFF_BASE * attempt)
                continue
            resp.raise_for_status()
            text = resp.json()["choices"][0]["message"]["content"].strip()
            time.sleep(REQUEST_SLEEP_SEC)
            return text
        except requests.exceptions.Timeout:
            time.sleep(API_BACKOFF_BASE * attempt)
        except Exception:
            time.sleep(API_BACKOFF_BASE * attempt)
    return ""

print("✓ API function ready")
print(f"  Planner  model : {GROQ_MODEL_PLANNER}")
print(f"  Executor model : {GROQ_MODEL_EXECUTOR}")

GROQ loaded: True
✓ API function ready
  Planner  model : meta-llama/llama-4-scout-17b-16e-instruct
  Executor model : meta-llama/llama-4-scout-17b-16e-instruct


## Data Preprocessing

In [4]:
def normalize_text(text: str) -> str:
    if pd.isna(text):
        return ""
    text = str(text).replace("\n", " ")
    return re.sub(r"\s+", " ", text).strip()

def parse_simple_to_list(value):
    if value is None:
        return [""]
    try:
        if pd.isna(value):
            return [""]
    except Exception:
        pass
    if isinstance(value, list):
        refs = value
    else:
        value = str(value).strip()
        if value in ("", "[]"):
            return [""]
        if value.startswith("[") and value.endswith("]"):
            try:
                parsed = ast.literal_eval(value)
                refs = parsed if isinstance(parsed, list) else [value]
            except Exception:
                refs = [value]
        else:
            refs = [value]
    clean = [normalize_text(x) for x in refs if normalize_text(x)]
    return clean if clean else [""]

def standardize_sentence_df(df: pd.DataFrame, with_labels: bool = True) -> pd.DataFrame:
    for col in ["pair_id", "para_id", "sent_id", "complex"]:
        if col not in df.columns:
            raise ValueError(f"Missing required column: {col}")
    out = df.copy()
    out["complex"] = out["complex"].apply(normalize_text)
    if with_labels:
        if "simple" not in out.columns:
            raise ValueError("Expected 'simple' column")
        if "label" not in out.columns:
            out["label"] = "unknown"
        out["references_parsed"] = out["simple"].apply(parse_simple_to_list)
        out["target_text"] = out["references_parsed"].apply(lambda xs: " ".join(xs).strip())
    return out

def add_next_sentence_context(df: pd.DataFrame) -> pd.DataFrame:
    out = df.copy().sort_values(["pair_id", "para_id", "sent_id"]).reset_index(drop=True)
    next_texts = []
    for i in range(len(out)):
        if i < len(out) - 1:
            same_para = (
                out.loc[i, "pair_id"] == out.loc[i+1, "pair_id"]
                and out.loc[i, "para_id"] == out.loc[i+1, "para_id"]
            )
            next_texts.append(normalize_text(out.loc[i+1, "complex"]) if same_para else "")
        else:
            next_texts.append("")
    out["next_sentence"] = next_texts
    return out

print("✓ Preprocessing ready")

✓ Preprocessing ready


In [5]:
train_df         = pd.read_csv(TRAIN_PATH)
valid_df         = pd.read_csv(VALID_PATH)
internal_test_df = pd.read_csv(TEST_INTERNAL_PATH)

train_df         = standardize_sentence_df(train_df,         with_labels=True)
valid_df         = standardize_sentence_df(valid_df,         with_labels=True)
internal_test_df = standardize_sentence_df(internal_test_df, with_labels=True)

train_df         = add_next_sentence_context(train_df)
valid_df         = add_next_sentence_context(valid_df)
internal_test_df = add_next_sentence_context(internal_test_df)

print("Train shape      :", train_df.shape)
print("Valid shape      :", valid_df.shape)
print("Internal test    :", internal_test_df.shape)

Train shape      : (11510, 13)
Valid shape      : (1697, 13)
Internal test    : (1512, 13)


## RAG Layer — MedSimplify Dictionary

In [11]:
import re
import pandas as pd

_medsimplify_df = pd.read_csv(MEDSIMPLIFY_PATH)

med_lookup = {}
for _, row in _medsimplify_df.iterrows():
    term = str(row["word"]).strip().lower()
    defn = normalize_text(str(row["definition"]))
    if term and defn:
        med_lookup[term] = defn

_sorted_terms = sorted(med_lookup.keys(), key=len, reverse=True)


RAG_STOP_TERMS = {
    "however", "therefore", "thus", "but", "and", "or", "although",
    "capacity", "measure", "difference", "evidence", "intervention",
    "result", "results", "change", "changes", "risk", "group",
    "groups", "study", "studies", "outcome", "outcomes",
    "evaluate", "effect", "effects", "reduce", "increase",
    "significant", "conclude", "benefit", "oral", "treatment",
    "treatments", "show", "report", "compare", "find", "found",
    "available", "review", "determine", "observe", "detect",
    "mouth", "potential", "moderate", "pregnancy", "regarding",
    "adverse", "complete", "follow-up", "long-term",
    "previous", "individual"
}


WEAK_SINGLE_TERMS = {
    "disease", "therapy", "reaction", "control", "dose", "symptoms",
    "screening", "incidence", "outcome", "outcomes", "pressure",
    "heart", "blood", "pain", "function", "management", "care"
}

def normalize_term_text(text: str) -> str:
    text = str(text).strip().lower()
    text = re.sub(r"\s+", " ", text)
    return text

def is_useful_term(term: str, definition: str) -> bool:
    term = normalize_term_text(term)
    definition = str(definition).strip()

    if not term or not definition:
        return False

    if term in RAG_STOP_TERMS:
        return False

   
    if len(term.split()) == 1 and len(term) <= 3:
        return False

    
    if len(term.split()) == 1 and term in WEAK_SINGLE_TERMS:
        return False

    
    if len(definition) < 5:
        return False

    return True


_multi_word_terms = [t for t in _sorted_terms if len(t.split()) >= 2]
_single_word_terms = [t for t in _sorted_terms if len(t.split()) == 1]

print(f"✓ MedSimplify loaded: {len(med_lookup)} terms")
print(f"  Sample terms: {list(med_lookup.keys())[:5]}")

def _match_terms_from_list(sentence_lower: str, terms: list, found: dict, occupied_spans: list, max_terms: int):
    for term in terms:
        if len(found) >= max_terms:
            break

        term = normalize_term_text(term)
        if term not in med_lookup:
            continue

        defn = med_lookup[term]

        if not is_useful_term(term, defn):
            continue

        pattern = r"\b" + re.escape(term) + r"\b"

        for match in re.finditer(pattern, sentence_lower):
            start, end = match.span()

            overlaps = any(not (end <= s or start >= e) for s, e in occupied_spans)
            if overlaps:
                continue

            found[term] = defn
            occupied_spans.append((start, end))
            break

def verify_rag_terms(found_terms: dict, sentence: str, max_terms: int = 2) -> dict:
    """
    فلترة ثانية قبل الحقن في الـ prompt.
    """
    verified = {}
    sentence_lower = normalize_term_text(sentence)

    for term, defn in found_terms.items():
        term = normalize_term_text(term)

        if term in RAG_STOP_TERMS:
            continue

        if len(term.split()) == 1 and term in WEAK_SINGLE_TERMS:
            continue

        if not re.search(r"\b" + re.escape(term) + r"\b", sentence_lower):
            continue

        verified[term] = defn

        if len(verified) >= max_terms:
            break

    return verified

def extract_terms(sentence: str, max_terms: int = RAG_MAX_TERMS) -> dict:
    """
    Extract medical terms from a sentence with:
    1) longest-first matching
    2) overlap blocking
    3) stop-term filtering
    4) preference for multi-word terms
    5) verification before prompt injection
    """
    sentence_lower = normalize_term_text(sentence)
    found = {}
    occupied_spans = []

    
    _match_terms_from_list(sentence_lower, _multi_word_terms, found, occupied_spans, max_terms)

    
    if len(found) < max_terms:
        _match_terms_from_list(sentence_lower, _single_word_terms, found, occupied_spans, max_terms)

    
    verified = verify_rag_terms(found, sentence, max_terms=min(max_terms, 2))
    return verified

def build_rag_context(found_terms: dict) -> str:
    if not found_terms:
        return ""

    lines = [
        "The following medical terms appear in this sentence.",
        "Use these definitions only to simplify difficult terms.",
        "Do not add new medical information.",
        "Do not over-explain.\n"
    ]

    for term, defn in found_terms.items():
        short_defn = defn if len(defn) <= 100 else defn[:100] + "..."
        lines.append(f"- {term}: {short_defn}")

    return "\n".join(lines)


test_sentences = [
    "The patient received nebulised salbutamol via MDI for COPD management.",
    "Corticosteroids may reduce the time to resolution of pleural effusion.",
    "Further high-quality research should evaluate the effect on neurodevelopmental outcomes.",
    "The patient was diagnosed with chronic obstructive pulmonary disease and was prescribed bronchodilator therapy."
]

for test_sent in test_sentences:
    test_terms = extract_terms(test_sent)
    print("=" * 80)
    print("Sentence:", test_sent)
    print("Found terms:", list(test_terms.keys()))
    print("RAG context:\n", build_rag_context(test_terms))

✓ MedSimplify loaded: 3123 terms
  Sample terms: ['bmi', 'bpg', 'chd', 'copd', 'cva']
Sentence: The patient received nebulised salbutamol via MDI for COPD management.
Found terms: ['copd']
RAG context:
 The following medical terms appear in this sentence.
Use these definitions only to simplify difficult terms.
Do not add new medical information.
Do not over-explain.

- copd: chronic obstructive pulmonary disease,
Sentence: Corticosteroids may reduce the time to resolution of pleural effusion.
Found terms: ['pleural effusion', 'corticosteroids']
RAG context:
 The following medical terms appear in this sentence.
Use these definitions only to simplify difficult terms.
Do not add new medical information.
Do not over-explain.

- pleural effusion: fluid in the chest cavity
- corticosteroids: asthma drug, asthma medicine
Sentence: Further high-quality research should evaluate the effect on neurodevelopmental outcomes.
Found terms: []
RAG context:
 
Sentence: The patient was diagnosed with chr

In [12]:
sample_df = train_df[["complex"]].sample(20, random_state=42).copy()

rows = []
for sent in sample_df["complex"]:
    found_terms = extract_terms(sent)
    rag_context = build_rag_context(found_terms)

    rows.append({
        "sentence": sent,
        "found_terms": list(found_terms.keys()),
        "n_terms": len(found_terms),
        "rag_context": rag_context
    })

rag_test_df = pd.DataFrame(rows)
display(rag_test_df)

,sentence,found_terms,n_terms,rag_context
0,"However, there was no clear evidence of a difference between interventions for all other measures of exercise capacity.",[],0,
1,Six trials used a split-mouth design.,[],0,
2,We found six publications involving a total of 47 participants requiring maxillary advancement of 4 mm to 10 mm.,[],0,
3,"As well as having potential benefits for apathy, methylphenidate probably also slightly improves cognition (MD 1.98, 95% CI 1.06 to 2.91, n = 145, 3 studies, moderate quality of evidence), and probably improves instrumental activities of daily living (MD 2.30, 95% CI 0.74 to 3.86, P = 0.004, n =...",[placebo],1,The following medical terms appear in this sentence.\nUse these definitions only to simplify difficult terms.\nDo not add new medical information.\nDo not over-explain.\n\n- placebo: sugar pill
4,Eleven trials administered probiotics to mothers during pregnancy and one trial administered probiotics to mothers after birth of their preterm infants.,[],0,
5,"Regarding secondary outcomes, there were no clear differences between the BCG alternating with IFN-α and BCG alone groups for disease-specific mortality (HR 2.74, 95% CI 0.73 to 10.28; 1 RCT; 205 participants; low-quality evidence), time-to-death (overall survival) (HR 1.00, 95% CI 0.68 to 1.47;...","[mortality, systemic]",2,"The following medical terms appear in this sentence.\nUse these definitions only to simplify difficult terms.\nDo not add new medical information.\nDo not over-explain.\n\n- mortality: death\n- systemic: throughout your body, in all parts of your body"
6,The moderate quality of the evidence for traumatic versus atraumatic needles suggests that further research is likely to have an important impact on our confidence in the estimate of effect.,"[atraumatic, traumatic]",2,The following medical terms appear in this sentence.\nUse these definitions only to simplify difficult terms.\nDo not add new medical information.\nDo not over-explain.\n\n- atraumatic: not damaging to tissue\n- traumatic: shocking
7,"In this update, we included seven trials with 1647 participants.",[],0,
8,"We judged the risks of selection and performance bias to be low; risks of detection bias, attrition bias, and reporting bias were unclear.",[],0,
9,No serious adverse events were reported with amantadine.,[],0,


## Evaluation Packages

In [13]:
def ensure_package(pkg_name):
    try:
        __import__(pkg_name)
    except ImportError:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pkg_name])

for pkg in ["evaluate", "sacrebleu", "textstat", "sacremoses"]:
    ensure_package(pkg)

import evaluate
from sacrebleu import sentence_bleu
from textstat import flesch_kincaid_grade

sari_metric = evaluate.load("sari")

def safe_fkgl(text: str) -> float:
    text = normalize_text(text)
    if not text:
        return np.nan
    try:
        return float(flesch_kincaid_grade(text))
    except Exception:
        return np.nan

def safe_sentence_bleu(pred: str, refs: list) -> float:
    pred = normalize_text(pred)
    refs = [normalize_text(r) for r in refs if normalize_text(r)]
    if not refs:
        refs = [""]
    try:
        return float(sentence_bleu(pred, refs).score) / 100.0
    except Exception:
        return np.nan

def safe_sentence_sari(src: str, pred: str, refs: list) -> float:
    src  = normalize_text(src)
    pred = normalize_text(pred)
    refs = [normalize_text(r) for r in refs if normalize_text(r)]
    if not refs:
        refs = [""]
    try:
        return float(sari_metric.compute(sources=[src], predictions=[pred], references=[refs])["sari"])
    except Exception:
        return np.nan

print("✓ Evaluation packages loaded")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 3.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 100.8/100.8 kB 5.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 177.1/177.1 kB 8.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.1/2.1 MB 73.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 897.5/897.5 kB 30.5 MB/s eta 0:00:00


✓ Evaluation packages loaded


## Execution Prompts V4 (مع RAG context)

In [14]:
# ============================================================
# Shared rules
# ============================================================
_SHARED_RULES = """
Critical rules:
- Preserve the original meaning exactly.
- Do NOT add information not in the original sentence.
- Do NOT introduce any entity, treatment, cause, or detail not explicitly mentioned in the original sentence.
- Do NOT use vague phrases like 'a certain treatment' or 'some kind of problem'.
- Preserve all important numbers, comparisons, and statistics unless the task explicitly allows removing secondary statistical detail.
- Use plain-language wording when it helps a general reader understand the sentence better.
- Replace difficult medical or scientific terms with simpler wording when possible without changing the meaning.
- If a technical term is important, you may keep it only if the sentence remains clear or if you make it easier to understand.
- Do NOT over-explain or turn one sentence into a long explanation unless the task explicitly asks for splitting.
- If you are unsure how to simplify, make only a minimal safe simplification.
- Output ONLY the final sentence(s), nothing else.
"""

# ============================================================
# RAG-aware prompts
# ============================================================
LIGHT_REPHRASE_PROMPT = """You are a medical text simplification expert.

Task: Rewrite this sentence in plainer English for a general reader.
Make it easier to understand WITHOUT changing its meaning or removing key facts.
Simplify difficult medical terms when helpful, not just the sentence structure.

{rag_context}
""" + _SHARED_RULES + """
Sentence:
{sentence}

Simplified:"""

SPLIT_REPHRASE_PROMPT = """You are a medical text simplification expert.

Task: Rewrite this sentence into 2-3 shorter, clearer sentences.
Each sentence must stand on its own and preserve the original meaning.
Use simpler wording where helpful, but do not add new information.

{rag_context}
""" + _SHARED_RULES + """
Sentence:
{sentence}

Simplified:"""

OMIT_DETAIL_PROMPT = """You are a medical text simplification expert.

Task: Rewrite this sentence by removing secondary statistical detail
such as p-values, confidence intervals, and secondary measurements.
Keep the main finding clearly stated and easy to understand.

{rag_context}
""" + _SHARED_RULES + """
Sentence:
{sentence}

Simplified:"""

SPLIT_AND_SIMPLIFY_PROMPT = """You are a medical text simplification expert.

Task: Rewrite this sentence into 2-3 shorter sentences AND simplify
technical wording where genuinely needed.
Use plain-language alternatives for difficult medical terms when possible.
Expand abbreviations only when necessary for a non-expert reader.

{rag_context}
""" + _SHARED_RULES + """
Sentence:
{sentence}

Simplified:"""

UNSURE_FALLBACK_PROMPT = """You are a conservative medical text simplification assistant.

Task: Produce the safest possible simplification of this sentence.
If the sentence is already clear, keep it unchanged or make only a very light simplification.

{rag_context}
""" + _SHARED_RULES + """
Sentence:
{sentence}

Simplified:"""

print("✓ Updated prompts ready")

✓ Updated prompts ready


## Semantic Planner V4

In [15]:
SEMANTIC_PLANNER_PROMPT = """You are a scientific text simplification planner.

Analyze the CURRENT sentence using the NEXT sentence as additional context.
Return ONLY valid JSON — no extra text, no markdown fences.

CURRENT SENTENCE:
{current_sentence}

NEXT SENTENCE (may be empty):
{next_sentence}

Field definitions:

is_already_simple:
  True if the sentence is already clear and understandable for a general reader
  and needs NO meaningful simplification. When in doubt, prefer false.

has_secondary_statistical_detail:
  True if the sentence mainly contains secondary numbers (p-values, confidence
  intervals, secondary measurements) that can be dropped without losing the
  main finding.

is_structurally_complex:
  True if the sentence has multiple clauses or ideas that would be clearer
  if split into shorter sentences.

contains_technical_terms:
  True if the sentence has medical/scientific terms a non-expert may not
  understand.

requires_abbreviation_expansion:
  True if important abbreviations or acronyms should be expanded.

preserve_numbers:
  True if important counts, percentages, or comparisons must be kept.

redundant_given_next_sentence:
  True if the NEXT sentence already states the same information more clearly,
  so the current sentence can be reduced or omitted.

depends_on_next_sentence:
  True if the current sentence is semantically incomplete without the next one.

Return exactly this JSON format:
{{
  "is_already_simple": false,
  "has_secondary_statistical_detail": false,
  "is_structurally_complex": false,
  "contains_technical_terms": true,
  "requires_abbreviation_expansion": false,
  "preserve_numbers": true,
  "redundant_given_next_sentence": false,
  "depends_on_next_sentence": false
}}
"""


DEFAULT_PLAN = {
    "is_already_simple": False,
    "has_secondary_statistical_detail": False,
    "is_structurally_complex": False,
    "contains_technical_terms": True,
    "requires_abbreviation_expansion": False,
    "preserve_numbers": True,
    "redundant_given_next_sentence": False,
    "depends_on_next_sentence": False,
}

def extract_first_json_block(text: str) -> str:
    text = str(text).strip()
    if not text:
        return ""
    if "```" in text:
        parts = text.split("```")
        candidates = [p.replace("json", "").strip() for p in parts if "{" in p and "}" in p]
        if candidates:
            text = candidates[0]
    start = text.find("{")
    end   = text.rfind("}")
    if start == -1 or end == -1 or end <= start:
        return ""
    return text[start:end+1]

def semantic_planner(sentence: str, next_sentence: str = "") -> tuple:
    """Returns: (plan_dict, planner_json_ok, used_default_plan)"""
    prompt = SEMANTIC_PLANNER_PROMPT.format(
        current_sentence=normalize_text(sentence),
        next_sentence=normalize_text(next_sentence)
    )
    raw = call_llama_api(prompt, max_tokens=160, use_large=True)
    try:
        json_text = extract_first_json_block(raw)
        if not json_text:
            return DEFAULT_PLAN.copy(), False, True
        data = json.loads(json_text)
        for k, v in DEFAULT_PLAN.items():
            if k not in data or not isinstance(data[k], bool):
                data[k] = v
        return data, True, False
    except Exception:
        return DEFAULT_PLAN.copy(), False, True

print("✓ Semantic planner ready")

✓ Semantic planner ready


## Strategy Mapping V4

In [16]:
def decide_strategy(plan: dict) -> str:
    if plan["is_already_simple"] and not plan["requires_abbreviation_expansion"]:
        return "KEEP_AS_IS"
    if plan["redundant_given_next_sentence"] and plan["has_secondary_statistical_detail"]:
        return "DELETE_OR_OMIT"
    if (
        plan["has_secondary_statistical_detail"]
        and not plan["is_structurally_complex"]
        and not plan["depends_on_next_sentence"]
        and not plan["contains_technical_terms"]
    ):
        return "DELETE_OR_OMIT"
    if plan["is_structurally_complex"] and (
        plan["contains_technical_terms"] or plan["requires_abbreviation_expansion"]
    ):
        return "SPLIT_AND_SIMPLIFY"
    if plan["is_structurally_complex"]:
        return "SPLIT"
    if not plan["is_already_simple"] and (
        plan["contains_technical_terms"] or plan["requires_abbreviation_expansion"]
    ):
        return "REPHRASE"
    return "UNSURE"

print("✓ Strategy mapping ready")

✓ Strategy mapping ready


## Execution Layer V4 (RAG-aware)

In [17]:
def clean_output(text: str) -> str:
    text = str(text).strip() if text else ""
    for prefix in ["output:", "answer:", "simplified:", "simplified sentence:",
                   "rewritten sentence:", "rewritten:"]:
        if text.lower().startswith(prefix):
            text = text[len(prefix):].strip()
            break
    return text.strip()

# ── Executor functions ──────────────

def run_light_rephrase(text: str, rag_context: str = "") -> str:
    prompt = LIGHT_REPHRASE_PROMPT.format(sentence=text, rag_context=rag_context)
    return clean_output(call_llama_api(prompt))

def run_split_rephrase(text: str, rag_context: str = "") -> str:
    prompt = SPLIT_REPHRASE_PROMPT.format(sentence=text, rag_context=rag_context)
    return clean_output(call_llama_api(prompt))

def run_omit_detail(text: str, rag_context: str = "") -> str:
    prompt = OMIT_DETAIL_PROMPT.format(sentence=text, rag_context=rag_context)
    return clean_output(call_llama_api(prompt))

def run_split_and_simplify(text: str, rag_context: str = "") -> str:
    prompt = SPLIT_AND_SIMPLIFY_PROMPT.format(sentence=text, rag_context=rag_context)
    return clean_output(call_llama_api(prompt))

def run_unsure_fallback(text: str, rag_context: str = "") -> str:
    prompt = UNSURE_FALLBACK_PROMPT.format(sentence=text, rag_context=rag_context)
    return clean_output(call_llama_api(prompt))

# ── Validate output ──────────────────────────────────────────

def count_sentences(text: str) -> int:
    parts = re.split(r'(?<=[.!?])\s+', text.strip())
    return len([p for p in parts if len(p.strip()) > 3])
def validate_output(strategy: str, output: str, original: str) -> tuple:
    """Returns: (validated_output, strategy_followed)"""
    output   = str(output).strip()   if output   else ""
    original = str(original).strip() if original else ""

    if strategy == "KEEP_AS_IS":
        return original, True

    if not output:
        return original, False

    orig_words = len(original.split())
    out_words  = len(output.split())

    if strategy == "SPLIT":
        return (output, True) if count_sentences(output) >= 2 else (original, False)

    if strategy == "SPLIT_AND_SIMPLIFY":
        return (output, True) if count_sentences(output) >= 2 else (original, False)

    if strategy == "DELETE_OR_OMIT":
        return (output, True) if out_words < orig_words else (original, False)

    if strategy == "REPHRASE":
        if out_words > orig_words * 1.6 or output == original:
            return original, False
        return output, True

    if out_words < 2:
        return original, False

    return output, True

# ── Execute strategy مع RAG ──────────────────────────────────

def execute_strategy(sentence: str, strategy: str, rag_context: str = "") -> tuple:
    """Returns: (output, execution_mode)"""
    strategy = str(strategy).strip().upper()

    if strategy == "KEEP_AS_IS":
        return sentence, "keep_as_is"

    if strategy == "DELETE_OR_OMIT":
        raw = run_omit_detail(sentence, rag_context)
        out, followed = validate_output(strategy, raw, sentence)
        return out, "normal_execution" if followed else "validation_fallback"

    if strategy == "SPLIT":
        raw = run_split_rephrase(sentence, rag_context)
        out, followed = validate_output(strategy, raw, sentence)
        return out, "normal_execution" if followed else "validation_fallback"

    if strategy == "SPLIT_AND_SIMPLIFY":
        raw = run_split_and_simplify(sentence, rag_context)
        out, followed = validate_output(strategy, raw, sentence)
        return out, "normal_execution" if followed else "validation_fallback"

    if strategy == "REPHRASE":
        raw = run_light_rephrase(sentence, rag_context)
        out, followed = validate_output(strategy, raw, sentence)
        return out, "normal_execution" if followed else "validation_fallback"

    if strategy == "UNSURE":
        raw = run_unsure_fallback(sentence, rag_context)
        out, _ = validate_output("REPHRASE", raw, sentence)
        return out, "unsure_fallback"

    return sentence, "unknown_strategy_fallback"

print("✓ Execution layer V4 ready")

✓ Execution layer V4 ready


## Pipeline V4 (مع RAG)

In [18]:
def run_pipeline_v4(df, text_col="complex", limit=200, desc="pipeline_v4"):
    """
    Pipeline V4:
    1) FKGL threshold → KEEP_AS_IS بدون API
    2) RAG lookup → استخراج المصطلحات
    3) Semantic planner → اختيار الاستراتيجية
    4) Execution + RAG context → تنفيذ التبسيط
    5) Validation + Diagnostic flags
    """
    if limit is not None and limit < len(df):
        work_df = df.sample(n=limit, random_state=SEED).copy()
    else:
        work_df = df.copy()

    rows = []
    print(f"\n Running Pipeline V4 (RAG) on {len(work_df)} sentences...")

    for i, (_, row) in enumerate(tqdm(work_df.iterrows(), total=len(work_df), desc=desc)):
        sentence      = normalize_text(row[text_col])
        next_sentence = normalize_text(row.get("next_sentence", ""))

        planner_json_ok   = None
        used_default_plan = None
        strategy_followed = None
        rag_terms_found   = 0
        rag_terms_list    = ""

        try:
            # ── Step 1: FKGL threshold ────────────────────────
            fkgl_src = safe_fkgl(sentence)
            if pd.notna(fkgl_src) and fkgl_src < FKGL_SIMPLE_THRESHOLD:
                plan              = DEFAULT_PLAN.copy()
                plan["is_already_simple"] = True
                strategy          = "KEEP_AS_IS"
                output            = sentence
                execution_mode    = "fkgl_threshold"
                planner_json_ok   = None
                used_default_plan = True
                strategy_followed = True
                # RAG مش ضروري هون
                rag_context       = ""

            else:
                # ── Step 2: RAG lookup ────────────────────────
                found_terms = extract_terms(sentence)
                rag_context = build_rag_context(found_terms)
                rag_terms_found = len(found_terms)
                rag_terms_list  = ", ".join(found_terms.keys())

                # ── Step 3: Semantic planner ──────────────────
                plan, planner_json_ok, used_default_plan = semantic_planner(
                    sentence, next_sentence
                )

                # ── Step 4: Strategy mapping ──────────────────
                strategy = decide_strategy(plan)

                # ── Step 5: Execution + RAG ───────────────────
                output, execution_mode = execute_strategy(
                    sentence, strategy, rag_context
                )
                strategy_followed = execution_mode in ("normal_execution", "keep_as_is")

        except Exception:
            plan              = {}
            strategy          = "UNSURE"
            output            = sentence
            execution_mode    = "exception_fallback"
            planner_json_ok   = False
            used_default_plan = True
            strategy_followed = False
            rag_context       = ""

        out_row = row.to_dict()
        out_row["plan"]              = json.dumps(plan, ensure_ascii=False)
        out_row["strategy"]          = strategy
        out_row["execution_mode"]    = execution_mode
        out_row["output"]            = output
        out_row["planner_json_ok"]   = planner_json_ok
        out_row["used_default_plan"] = used_default_plan
        out_row["output_changed"]    = (output != sentence)
        out_row["strategy_followed"] = strategy_followed
        out_row["rag_terms_found"]   = rag_terms_found
        out_row["rag_terms"]         = rag_terms_list
        out_row["output_word_count"] = len(output.split())
        out_row["source_word_count"] = len(sentence.split())

        rows.append(out_row)

        if (i + 1) % SAVE_EVERY == 0:
            pd.DataFrame(rows).to_csv(
                OUTPUT_DIR / f"{desc}_checkpoint.csv", index=False
            )

    result_df = pd.DataFrame(rows)
    result_df.to_csv(OUTPUT_DIR / f"{desc}_predictions.csv", index=False)
    print(f"✓ Pipeline V4 finished — {len(result_df)} rows")
    return result_df

print("✓ Pipeline V4 ready")

✓ Pipeline V4 ready


## Evaluation Layer

In [19]:
def evaluate_outputs(pred_df: pd.DataFrame) -> pd.DataFrame:
    rows = []
    print("\n Evaluating predictions...")
    for _, row in tqdm(pred_df.iterrows(), total=len(pred_df), desc="Evaluating"):
        refs = (row["references_parsed"]
                if "references_parsed" in row and row["references_parsed"]
                else parse_simple_to_list(row.get("simple", "")))
        src  = normalize_text(row["complex"])
        pred = normalize_text(row["output"])
        fkgl_src  = safe_fkgl(src)
        fkgl_pred = safe_fkgl(pred)
        out = row.to_dict()
        out["references_parsed"] = refs
        out["num_references"]    = len(refs)
        out["sari"]              = safe_sentence_sari(src, pred, refs)
        out["bleu"]              = safe_sentence_bleu(pred, refs)
        out["fkgl_source"]       = fkgl_src
        out["fkgl_prediction"]   = fkgl_pred
        out["fkgl_delta"]        = (fkgl_pred - fkgl_src
                                    if pd.notna(fkgl_src) and pd.notna(fkgl_pred) else np.nan)
        out["compression_ratio"] = (len(pred.split()) / max(1, len(src.split()))
                                    if src else np.nan)
        rows.append(out)
    return pd.DataFrame(rows)

def print_evaluation_report(eval_df: pd.DataFrame, title="EVALUATION REPORT"):
    print("\n" + "="*70)
    print(title)
    print("="*70)
    print(f"\n📈 Overall Metrics:")
    print(f"  Total sentences     : {len(eval_df)}")
    print(f"  Overall SARI        : {eval_df['sari'].mean():.4f}")
    print(f"  Overall BLEU        : {eval_df['bleu'].mean():.4f}")
    print(f"  Avg FKGL (source)   : {eval_df['fkgl_source'].mean():.2f}")
    print(f"  Avg FKGL (pred)     : {eval_df['fkgl_prediction'].mean():.2f}")
    print(f"  Avg FKGL delta      : {eval_df['fkgl_delta'].mean():.2f} (negative = easier)")
    print(f"  Avg compression     : {eval_df['compression_ratio'].mean():.3f}")

    if "planner_json_ok" in eval_df.columns:
        print(f"\n Diagnostic Flags:")
        print(f"  Planner JSON OK     : {eval_df['planner_json_ok'].sum()} / {len(eval_df)}")
        print(f"  Used default plan   : {eval_df['used_default_plan'].sum()} / {len(eval_df)}")
        print(f"  Output changed      : {eval_df['output_changed'].sum()} / {len(eval_df)}")
        print(f"  Strategy followed   : {eval_df['strategy_followed'].sum()} / {len(eval_df)}")

    if "rag_terms_found" in eval_df.columns:
        n_with_rag = (eval_df["rag_terms_found"] > 0).sum()
        avg_terms  = eval_df["rag_terms_found"].mean()
        print(f"\n RAG Stats:")
        print(f"  Sentences with RAG terms : {n_with_rag} / {len(eval_df)}")
        print(f"  Avg terms per sentence   : {avg_terms:.2f}")

        
        with_rag    = eval_df[eval_df["rag_terms_found"] > 0]
        without_rag = eval_df[eval_df["rag_terms_found"] == 0]
        if len(with_rag) > 0:
            print(f"  SARI (with RAG terms)    : {with_rag['sari'].mean():.4f} (n={len(with_rag)})")
        if len(without_rag) > 0:
            print(f"  SARI (no RAG terms)      : {without_rag['sari'].mean():.4f} (n={len(without_rag)})")

    print(f"\n Per-Strategy Performance:")
    for strat in ["KEEP_AS_IS", "DELETE_OR_OMIT", "REPHRASE", "SPLIT", "SPLIT_AND_SIMPLIFY", "UNSURE"]:
        subset = eval_df[eval_df["strategy"] == strat]
        if len(subset) > 0:
            print(f"\n  {strat}:")
            print(f"    Count         : {len(subset)}")
            print(f"    SARI          : {subset['sari'].mean():.4f}")
            print(f"    BLEU          : {subset['bleu'].mean():.4f}")
            if "strategy_followed" in subset.columns:
                print(f"    Followed      : {subset['strategy_followed'].sum()}/{len(subset)}")
    print("\n" + "="*70 + "\n")

print("✓ Evaluation layer ready")

✓ Evaluation layer ready


## Pretty Display

In [20]:
def show_strategy_examples(
    eval_df: pd.DataFrame,
    title="RESULTS",
    strategies=None,
    best_n=3,
    worst_n=3,
    text_limit=300
):
    def short(x):
        x = "" if pd.isna(x) else str(x).replace("\n", " ").strip()
        return x if len(x) <= text_limit else x[:text_limit] + "..."

    if strategies is None:
        strategies = sorted(eval_df["strategy"].dropna().unique())

    print("\n" + "="*90)
    print(title)
    print("="*90)

    for strat in strategies:
        subset = eval_df[eval_df["strategy"] == strat].copy()
        print("\n" + "="*90)
        print(f"STRATEGY: {strat}")
        print("="*90)
        if len(subset) == 0:
            print("  No examples.")
            continue

        print(f"\n📊 Stats: count={len(subset)} | "
              f"SARI={subset['sari'].mean():.4f} | "
              f"BLEU={subset['bleu'].mean():.4f} | "
              f"FKGL Δ={subset['fkgl_delta'].mean():+.2f} | "
              f"Compression={subset['compression_ratio'].mean():.3f}")

        for label, data in [(" BEST", subset.nlargest(best_n, "sari")),
                            (" WORST", subset.nsmallest(worst_n, "sari"))]:
            print(f"\n{label} EXAMPLES")
            print("-"*90)
            for i, (_, row) in enumerate(data.iterrows(), 1):
                print(f"\n{i}. SARI: {row.get('sari', np.nan):.4f} | "
                      f"BLEU: {row.get('bleu', np.nan):.4f} | "
                      f"FKGL Δ: {row.get('fkgl_delta', np.nan):+.2f} | "
                      f"Compression: {row.get('compression_ratio', np.nan):.3f}")
                print(f"   Exec mode      : {row.get('execution_mode', '')}")
                print(f"   RAG terms      : {row.get('rag_terms', '')}")
                print(f"   Strategy follow: {row.get('strategy_followed', '')}")
                print(f"   INPUT  : {short(row.get('complex', ''))}")
                print(f"   NEXT   : {short(row.get('next_sentence', ''))}")
                print(f"   PLAN   : {short(row.get('plan', ''))}")
                print(f"   OUTPUT : {short(row.get('output', ''))}")
                print(f"   GOLD   : {short(row.get('simple', ''))}")

    print("\n" + "="*90)
    print("✓ Analysis complete")
    print("="*90)

print("✓ Display function ready")

✓ Display function ready


## Train — Run + Evaluate

In [16]:
train_preds = run_pipeline_v4(
    train_df, text_col="complex", limit=SPLIT_LIMIT, desc="train_v4"
)

train_eval = evaluate_outputs(train_preds)
train_eval.to_csv(OUTPUT_DIR / "train_v4_evaluation.csv", index=False)

print("\nTRAIN STRATEGY DISTRIBUTION")
print(train_preds["strategy"].value_counts(dropna=False))
print("\nTRAIN EXECUTION MODE DISTRIBUTION")
print(train_preds["execution_mode"].value_counts(dropna=False))

print_evaluation_report(train_eval, title="TRAIN EVALUATION REPORT")
show_strategy_examples(train_eval, title="TRAIN — BEST AND WORST EXAMPLES", best_n=3, worst_n=3)


🔄 Running Pipeline V4 (RAG) on 200 sentences...


train_v4:   0%|          | 0/200 [00:00<?, ?it/s]

✓ Pipeline V4 finished — 200 rows

📊 Evaluating predictions...


Evaluating:   0%|          | 0/200 [00:00<?, ?it/s]


TRAIN STRATEGY DISTRIBUTION
strategy
REPHRASE              71
SPLIT_AND_SIMPLIFY    63
KEEP_AS_IS            56
UNSURE                 6
SPLIT                  4
Name: count, dtype: int64

TRAIN EXECUTION MODE DISTRIBUTION
execution_mode
normal_execution       118
fkgl_threshold          56
validation_fallback     20
unsure_fallback          6
Name: count, dtype: int64

TRAIN EVALUATION REPORT

📈 Overall Metrics:
  Total sentences     : 200
  Overall SARI        : 43.2297
  Overall BLEU        : 0.1201
  Avg FKGL (source)   : 13.32
  Avg FKGL (pred)     : 10.82
  Avg FKGL delta      : -2.51 (negative = easier)
  Avg compression     : 1.057

🔧 Diagnostic Flags:
  Planner JSON OK     : 144 / 200
  Used default plan   : 56 / 200
  Output changed      : 118 / 200
  Strategy followed   : 174 / 200

📚 RAG Stats:
  Sentences with RAG terms : 112 / 200
  Avg terms per sentence   : 1.31
  SARI (with RAG terms)    : 38.5841 (n=112)
  SARI (no RAG terms)      : 49.1424 (n=88)

🔄 Per-Strategy Per

## Valid — Run + Evaluate

In [21]:
valid_preds = run_pipeline_v4(
    valid_df, text_col="complex", limit=SPLIT_LIMIT, desc="valid_v4"
)

valid_eval = evaluate_outputs(valid_preds)
valid_eval.to_csv(OUTPUT_DIR / "valid_v4_evaluation.csv", index=False)

print("\nVALID STRATEGY DISTRIBUTION")
print(valid_preds["strategy"].value_counts(dropna=False))
print("\nVALID EXECUTION MODE DISTRIBUTION")
print(valid_preds["execution_mode"].value_counts(dropna=False))

print_evaluation_report(valid_eval, title="VALID EVALUATION REPORT")
show_strategy_examples(valid_eval, title="VALID — BEST AND WORST EXAMPLES", best_n=3, worst_n=3)


🔄 Running Pipeline V4 (RAG) on 200 sentences...


valid_v4:   0%|          | 0/200 [00:00<?, ?it/s]

✓ Pipeline V4 finished — 200 rows

📊 Evaluating predictions...


Evaluating:   0%|          | 0/200 [00:00<?, ?it/s]


VALID STRATEGY DISTRIBUTION
strategy
REPHRASE              76
SPLIT_AND_SIMPLIFY    66
KEEP_AS_IS            57
UNSURE                 1
Name: count, dtype: int64

VALID EXECUTION MODE DISTRIBUTION
execution_mode
normal_execution       126
fkgl_threshold          57
validation_fallback     16
unsure_fallback          1
Name: count, dtype: int64

VALID EVALUATION REPORT

📈 Overall Metrics:
  Total sentences     : 200
  Overall SARI        : 43.7182
  Overall BLEU        : 0.1373
  Avg FKGL (source)   : 13.00
  Avg FKGL (pred)     : 10.36
  Avg FKGL delta      : -2.64 (negative = easier)
  Avg compression     : 1.065

🔧 Diagnostic Flags:
  Planner JSON OK     : 143 / 200
  Used default plan   : 57 / 200
  Output changed      : 126 / 200
  Strategy followed   : 183 / 200

📚 RAG Stats:
  Sentences with RAG terms : 120 / 200
  Avg terms per sentence   : 1.33
  SARI (with RAG terms)    : 39.2376 (n=120)
  SARI (no RAG terms)      : 50.4391 (n=80)

🔄 Per-Strategy Performance:

  KEEP_AS_IS:


## Internal Test — Run + Evaluate

In [22]:
internal_test_preds = run_pipeline_v4(
    internal_test_df, text_col="complex", limit=SPLIT_LIMIT, desc="internal_test_v4"
)

internal_test_eval = evaluate_outputs(internal_test_preds)
internal_test_eval.to_csv(OUTPUT_DIR / "internal_test_v4_evaluation.csv", index=False)

print("\nINTERNAL TEST STRATEGY DISTRIBUTION")
print(internal_test_preds["strategy"].value_counts(dropna=False))
print("\nINTERNAL TEST EXECUTION MODE DISTRIBUTION")
print(internal_test_preds["execution_mode"].value_counts(dropna=False))

print_evaluation_report(internal_test_eval, title="INTERNAL TEST EVALUATION REPORT")
show_strategy_examples(internal_test_eval, title="INTERNAL TEST — BEST AND WORST EXAMPLES", best_n=3, worst_n=3)


🔄 Running Pipeline V4 (RAG) on 200 sentences...


internal_test_v4:   0%|          | 0/200 [00:00<?, ?it/s]

✓ Pipeline V4 finished — 200 rows

📊 Evaluating predictions...


Evaluating:   0%|          | 0/200 [00:00<?, ?it/s]


INTERNAL TEST STRATEGY DISTRIBUTION
strategy
REPHRASE              107
KEEP_AS_IS             62
SPLIT_AND_SIMPLIFY     24
UNSURE                  5
SPLIT                   2
Name: count, dtype: int64

INTERNAL TEST EXECUTION MODE DISTRIBUTION
execution_mode
validation_fallback    90
fkgl_threshold         62
normal_execution       43
unsure_fallback         5
Name: count, dtype: int64

INTERNAL TEST EVALUATION REPORT

📈 Overall Metrics:
  Total sentences     : 200
  Overall SARI        : 51.3574
  Overall BLEU        : 0.1734
  Avg FKGL (source)   : 13.02
  Avg FKGL (pred)     : 12.07
  Avg FKGL delta      : -0.95 (negative = easier)
  Avg compression     : 1.023

🔧 Diagnostic Flags:
  Planner JSON OK     : 64 / 200
  Used default plan   : 136 / 200
  Output changed      : 44 / 200
  Strategy followed   : 105 / 200

📚 RAG Stats:
  Sentences with RAG terms : 109 / 200
  Avg terms per sentence   : 1.27
  SARI (with RAG terms)    : 49.1923 (n=109)
  SARI (no RAG terms)      : 53.9508 (n

In [25]:
import requests

resp = requests.post(
    "https://api.groq.com/openai/v1/chat/completions",
    headers={
        "Authorization": f"Bearer {GROQ_API_KEY}",
        "Content-Type": "application/json",
    },
    json={
        "model": "meta-llama/llama-4-scout-17b-16e-instruct",
        "messages": [{"role": "user", "content": "Say hello"}],
        "max_tokens": 10,
        "temperature": 0.0,
    },
    timeout=15
)

print("Status                    :", resp.status_code)
print("Remaining requests today  :", resp.headers.get("x-ratelimit-remaining-requests"))
print("Remaining tokens/minute   :", resp.headers.get("x-ratelimit-remaining-tokens"))
print("Reset requests in         :", resp.headers.get("x-ratelimit-reset-requests"))
print("Body                      :", resp.json())

Status                    : 200
Remaining requests today  : 4
Remaining tokens/minute   : 29978
Reset requests in         : 23h54m14.4s
Body                      : {'id': 'chatcmpl-86e29135-cda2-4893-9eed-04e0140d9b27', 'object': 'chat.completion', 'created': 1776681536, 'model': 'meta-llama/llama-4-scout-17b-16e-instruct', 'choices': [{'index': 0, 'message': {'role': 'assistant', 'content': 'Hello! How can I help you today?'}, 'logprobs': None, 'finish_reason': 'length'}], 'usage': {'queue_time': 0.429563067, 'prompt_tokens': 12, 'prompt_time': 0.000213478, 'completion_tokens': 10, 'completion_time': 0.022610101, 'total_tokens': 22, 'total_time': 0.022823579}, 'usage_breakdown': None, 'system_fingerprint': 'fp_5bd1e7ad73', 'x_groq': {'id': 'req_01kpn7hjr5fez9sw3daeh2b083', 'seed': 1281363028}, 'service_tier': 'on_demand'}


In [21]:
# ============================================================
# SECTION: Gradio Interface V2 — Full Diagnostic Display
# ============================================================

def ensure_gradio():
    try:
        __import__("gradio")
    except ImportError:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "gradio"])

ensure_gradio()
import gradio as gr

def simplify_sentence(sentence: str):
    sentence = normalize_text(sentence)

    if not sentence:
        return "", "-", "-", "-", "-", "-", "-", "-"

    # ── Step 1: FKGL check ────────────────────────────────
    fkgl_src = safe_fkgl(sentence)
    if pd.notna(fkgl_src) and fkgl_src < FKGL_SIMPLE_THRESHOLD:
        return (
            sentence,                          # Simplified Output
            "KEEP_AS_IS",                      # Strategy
            "fkgl_threshold",                  # Execution Mode
            f"{fkgl_src:.1f} → {fkgl_src:.1f} (no change needed)",  # Readability
            "None",                            # RAG Terms Found
            "{}",                              # Plan JSON
            "N/A (FKGL threshold)",            # Planner JSON OK
            "Yes (FKGL threshold)",            # Used Default Plan
        )

    # ── Step 2: RAG ───────────────────────────────────────
    found_terms = extract_terms(sentence)
    rag_context = build_rag_context(found_terms)
    rag_info    = ", ".join(found_terms.keys()) if found_terms else "None"

    # ── Step 3: Planner ───────────────────────────────────
    plan, planner_json_ok, used_default = semantic_planner(sentence, "")
    strategy = decide_strategy(plan)

    # ── Step 4: Execute ───────────────────────────────────
    output, exec_mode = execute_strategy(sentence, strategy, rag_context)

    # ── Step 5: FKGL after ────────────────────────────────
    fkgl_pred = safe_fkgl(output)
    fkgl_info = (
        f"{fkgl_src:.1f} → {fkgl_pred:.1f} "
        f"({'easier ✅' if fkgl_pred < fkgl_src else 'same ➡️'})"
        if pd.notna(fkgl_src) and pd.notna(fkgl_pred)
        else "N/A"
    )

    # ── Format plan JSON ──────────────────────────────────
    try:
        plan_display = json.dumps(plan, indent=2)
    except Exception:
        plan_display = str(plan)

    return (
        output,                                          # Simplified Output
        strategy,                                        # Strategy
        exec_mode,                                       # Execution Mode
        fkgl_info,                                       # Readability
        rag_info,                                        # RAG Terms Found
        plan_display,                                    # Plan JSON
        "✅ Yes" if planner_json_ok else "❌ No",        # Planner JSON OK
        "Yes" if used_default else "No",                 # Used Default Plan
    )


# ── Examples ──────────────────────────────────────────────────
examples = [
    ["Nebulised salbutamol was administered via metered-dose inhaler."],
    ["The intervention showed no statistically significant effect on mortality."],
    ["Further high-quality research should evaluate the effect on neurodevelopmental outcomes."],
    ["Patients with COPD showed improved FEV1 following bronchodilator therapy."],
    ["The meta-analysis showed a significant reduction in systolic blood pressure."],
]

# ── Interface ──────────────────────────────────────────────────
with gr.Blocks(title="Medical Text Simplifier V4") as demo:

    gr.Markdown("""
    # 🏥 Medical Text Simplifier V4
    **Powered by RAG + MedSimplify Dictionary + Scout Planner**
    Enter a complex medical sentence — the system analyzes, plans, and simplifies automatically.
    """)

    # ── Input + Output ────────────────────────────────────
    with gr.Row():
        with gr.Column(scale=2):
            input_box = gr.Textbox(
                label="📝 Complex Medical Sentence",
                placeholder="Enter a medical sentence to simplify...",
                lines=4
            )
            submit_btn = gr.Button("🔍 Simplify", variant="primary", size="lg")

        with gr.Column(scale=2):
            output_box = gr.Textbox(
                label="✅ Simplified Output",
                lines=4,
                interactive=False
            )

    # ── Main metrics ──────────────────────────────────────
    with gr.Row():
        strategy_box = gr.Textbox(
            label="📋 Strategy",
            interactive=False
        )
        exec_mode_box = gr.Textbox(
            label="⚙️ Execution Mode",
            interactive=False
        )
        fkgl_box = gr.Textbox(
            label="📊 Readability",
            interactive=False
        )

    # ── RAG + Planner diagnostics ─────────────────────────
    with gr.Row():
        rag_box = gr.Textbox(
            label="📚 RAG Terms Found",
            interactive=False
        )
        planner_ok_box = gr.Textbox(
            label="🧠 Planner JSON OK",
            interactive=False
        )
        default_plan_box = gr.Textbox(
            label="🔄 Used Default Plan",
            interactive=False
        )

    # ── Plan JSON ─────────────────────────────────────────
    plan_box = gr.Textbox(
        label="🗂️ Plan JSON",
        lines=10,
        interactive=False
    )

    # ── Examples ──────────────────────────────────────────
    gr.Examples(
        examples=examples,
        inputs=input_box,
        label="💡 Try these examples"
    )

    # ── Button action ─────────────────────────────────────
    submit_btn.click(
        fn=simplify_sentence,
        inputs=input_box,
        outputs=[
            output_box,
            strategy_box,
            exec_mode_box,
            fkgl_box,
            rag_box,
            plan_box,
            planner_ok_box,
            default_plan_box,
        ]
    )

demo.launch(share=True)

* Running on local URL:  http://127.0.0.1:7860
* Running on public URL: https://8df3d296e080a82354.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


## Official Test — Submission File

In [ ]:
# ============================================================
# تشغيل على الـ official test JSON وبناء ملف الـ submission
# ============================================================

TEAM_ID  = "raneemrabi3"
TASK_ID  = "task11"
METHOD   = "SemanticSelectorV4RAG"
RUN_ID_S = f"{TEAM_ID}_{TASK_ID}_{METHOD}"

# تحميل الـ official test
with open(OFFICIAL_TEST_JSON, encoding="utf-8") as f:
    official_test_data = json.load(f)

official_test_df = pd.DataFrame(official_test_data)
official_test_df["complex"] = official_test_df["complex"].apply(normalize_text)
official_test_df = add_next_sentence_context(official_test_df)

print(f"Official test size: {len(official_test_df)}")

# تشغيل Pipeline V4 على كامل الـ official test
official_preds = run_pipeline_v4(
    official_test_df,
    text_col="complex",
    limit=None,  # كل الجمل
    desc="official_test_v4"
)

# بناء ملف الـ submission
def build_submission(pred_df: pd.DataFrame, run_id: str) -> list:
    records = []
    for _, row in pred_df.iterrows():
        records.append({
            "pair_id"   : row["pair_id"],
            "para_id"   : row["para_id"],
            "sent_id"   : row["sent_id"],
            "complex"   : row["complex"],
            "prediction": row["output"],
            "run_id"    : run_id
        })
    return records

submission_records = build_submission(official_preds, RUN_ID_S)
submission_path = OUTPUT_DIR / f"{RUN_ID_S}.json"

with open(submission_path, "w", encoding="utf-8") as f:
    json.dump(submission_records, f, ensure_ascii=False, indent=2)

print(f"\n✓ Submission saved: {submission_path}")
print(f"  Run ID   : {RUN_ID_S}")
print(f"  Records  : {len(submission_records)}")
print(f"  Sample   :")
print(json.dumps(submission_records[0], indent=4, ensure_ascii=False))